In [ ]:
# ---- Q1. Simulation size: is there a column that records it? (QBE sandbox)
display(pd.read_sql("""
    SELECT TABLE_SCHEMA, TABLE_NAME, COLUMN_NAME
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'SIM'
      AND (COLUMN_NAME LIKE '%SIM%' OR COLUMN_NAME LIKE '%YEAR%'
           OR COLUMN_NAME LIKE '%ITER%' OR COLUMN_NAME LIKE '%NUM%')
    ORDER BY TABLE_NAME, COLUMN_NAME
""", engine_ref))

# ---- Q2. Where do the unmapped group events live? (QBE sandbox)
mapped = set(mapping_df["VENDOREVENTID"])
g = group_elt.assign(AAL=group_elt["RATE"] * group_elt["PERSPVALUE"])
miss = g[~g["VENDOREVENTID"].isin(mapped)]
print(f"unmapped: {len(miss):,} events | AAL lost "
      f"{miss['AAL'].sum() / g['AAL'].sum():.1%}")
ids = ",".join(map(str, miss["VENDOREVENTID"].unique()))
display(pd.read_sql(f"""
    SELECT HAZARDZONEID, VENDORVERID, COUNT(DISTINCT VENDOREVENTID) AS n_events
    FROM EVT.VENDOREVENTSMAP
    WHERE VENDOREVENTID IN ({ids})
    GROUP BY HAZARDZONEID, VENDORVERID
    ORDER BY n_events DESC
""", engine_ref))

# ---- Q3. Account universe: which filter drops accounts? (RMS)
display(pd.read_sql("""
    SELECT PERSPCODE, COUNT(DISTINCT ID) AS n_accounts
    FROM DBO.RDM_ACCOUNT
    WHERE ANLSID = 112
    GROUP BY PERSPCODE
""", engine_rdm))

# ---- Q4. Group ELT: is there a perspective column we haven't seen? (QBE main)
display(pd.read_sql("""
    SELECT COLUMN_NAME
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'MARG'
      AND TABLE_NAME   = 'GROUP_ELT_202604_MPML_NAWS'
    ORDER BY ORDINAL_POSITION
""", engine_qbe))

In [ ]:
display(pd.read_sql("SELECT TOP 5 * FROM SIM.YEQTInfo", engine_ref))     # check the YEQTID column name

display(pd.read_sql("""
    SELECT yi.*, sti.NumSimID, sti.SimTableType
    FROM SIM.YEQTInfo yi
    INNER JOIN SIM.SimTableInfo sti ON sti.SimTableID = yi.SimTableID
    WHERE yi.YEQTID = 69
""", engine_ref))

In [ ]:
def load_n_sims(yeqtid=YEQTID):
    n = int(pd.read_sql(f"""
        SELECT sti.NumSimID
        FROM SIM.YEQTInfo yi
        INNER JOIN SIM.SimTableInfo sti ON sti.SimTableID = yi.SimTableID
        WHERE yi.YEQTID = {yeqtid}
    """, engine_ref).iloc[0, 0])
    print(f"YEQTID {yeqtid}: {n:,} simulated years (from SimTableInfo)")
    return n

N_SIMS = load_n_sims()
TARGET_RANK = round(N_SIMS / RETURN_PERIOD)          # expect 2,500
BAND_LO, BAND_HI = TARGET_RANK - 1500, TARGET_RANK + 1500